# 📊 Análise de Inadimplência em Cartões de Crédito

**Contexto do Projeto:**  
Nosso cliente é uma empresa de cartão de crédito que nos forneceu um dataset com dados demográficos e financeiros de **30.000 titulares de contas** referentes aos últimos seis meses. Cada linha representa uma conta de crédito individual.

**Objetivo:**  
As linhas são rotuladas de acordo com se, no mês seguinte ao período histórico de seis meses, o titular ficou **inadimplente** (não realizou o pagamento mínimo). O objetivo final é entender o perfil de inadimplência e, futuramente, construir um modelo preditivo.

**Dataset:** `default_credit_card_clients.csv`

---

## 📁 Estrutura do Notebook
1. Importação de bibliotecas e carregamento dos dados
2. Exploração inicial — colunas e linhas
3. Tipos de características (categóricas vs numéricas)
4. Aparência dos dados (distribuições e frequências)
5. Verificação de dados faltantes

## 1. Importação de Bibliotecas e Carregamento dos Dados

Começamos importando as bibliotecas necessárias para análise de dados e carregando o dataset em um DataFrame do pandas.

In [ ]:
import pandas as pd
import numpy as np

# Carregamento do dataset
df = pd.read_csv(r'C:\Users\BG-CD001\Downloads\default_credit_card_clients.csv')

print('✅ Dataset carregado com sucesso!')
print(f'   Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas')

---

## 2. Exploração Inicial

### 2.1 Quantas colunas os dados contêm?

As colunas de um dataset podem ser de três tipos:
- **Características (features):** variáveis usadas para análise ou previsão
- **Resposta (target):** variável que queremos prever (`default payment next month`)
- **Metadados:** identificadores que não entram na modelagem (ex: `ID`)

In [ ]:
print(f'Número de colunas: {df.shape[1]}')
print(f'\nNomes das colunas:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2}. {col}')

> **Resultado esperado:** 25 colunas — 1 metadado (`ID`), 23 características e 1 variável resposta (`default payment next month`).

### 2.2 Quantas linhas (amostras)?

A definição de linha é fundamental: neste dataset, **cada linha representa uma conta de crédito**, ou seja, um titular único.

In [ ]:
print(f'Número de linhas (amostras): {df.shape[0]:,}')
print(f'Cada linha representa: um titular de conta de cartão de crédito')

> **Resultado esperado:** 30.000 linhas — uma por titular de conta.

---

## 3. Tipos de Características

### O que são características categóricas e numéricas?

- **Categóricas:** valores que pertencem a classes discretas (ex: `SEX`: 1=Masculino, 2=Feminino)
- **Numéricas contínuas:** valores em escala numérica (ex: `LIMIT_BAL`: limite de crédito em dólares)

Abaixo verificamos os tipos de dados de cada coluna:

In [ ]:
print('Tipos de dados por coluna:')
print(df.dtypes)
print(f'\nTotal de colunas numéricas (int64/float64): {df.select_dtypes(include=np.number).shape[1]}')
print(f'Total de colunas texto (object): {df.select_dtypes(include="object").shape[1]}')

### Classificação manual das colunas

Analisando o significado de cada coluna, podemos classificá-las como:

| Grupo | Colunas | Descrição |
|---|---|---|
| Metadado | `ID` | Identificador único da conta |
| Categórica | `SEX` | 1=Masculino, 2=Feminino |
| Categórica | `EDUCATION` | 1=Pós-grad, 2=Universidade, 3=Ensino médio, 4=Outros |
| Categórica | `MARRIAGE` | 1=Casado, 2=Solteiro, 3=Outro |
| Categórica | `PAY_1` a `PAY_6` | Status de pagamento dos últimos 6 meses |
| Numérica | `LIMIT_BAL` | Limite de crédito (em dólares) |
| Numérica | `AGE` | Idade do titular |
| Numérica | `BILL_AMT1` a `BILL_AMT6` | Valor da fatura dos últimos 6 meses |
| Numérica | `PAY_AMT1` a `PAY_AMT6` | Valor pago nos últimos 6 meses |
| **Resposta** | `default payment next month` | **0=Não inadimplente, 1=Inadimplente** |

In [ ]:
# Verificando os valores únicos das colunas categóricas
categoricas = ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

print('Valores únicos por coluna categórica:\n')
for col in categoricas:
    unicos = sorted(df[col].astype(str).unique().tolist())
    print(f'  {col}: {unicos}')

---

## 4. Aparência dos Dados

### 4.1 Resumo estatístico das variáveis numéricas

O método `.describe()` retorna estatísticas descritivas como média, desvio padrão, mínimo, máximo e quartis para cada coluna numérica.

In [ ]:
# Estatísticas descritivas das colunas numéricas
df.describe()

### 4.2 Frequência das variáveis categóricas

Para variáveis categóricas, o mais informativo é ver quantas ocorrências existem de cada classe.

In [ ]:
print('=== SEX (1=Masculino, 2=Feminino) ===')
print(df['SEX'].value_counts())

print('\n=== EDUCATION (1=Pós-grad, 2=Universidade, 3=Ensino Médio, 4=Outros) ===')
print(df['EDUCATION'].value_counts())

print('\n=== MARRIAGE (1=Casado, 2=Solteiro, 3=Outro) ===')
print(df['MARRIAGE'].value_counts())

In [ ]:
# Distribuição da variável resposta (target)
print('=== TARGET: default payment next month ===')
print(df['default payment next month'].value_counts())
print()
print('Proporção:')
print(df['default payment next month'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

> **Observação importante:** A variável resposta é **desbalanceada** — aproximadamente 78% dos titulares não são inadimplentes e 22% são. Isso é relevante para futuras etapas de modelagem.

---

## 5. Verificação de Dados Faltantes

Dados faltantes podem ser **explícitos** (NaN/null) ou **implícitos** (valores inválidos como `0` em uma coluna onde não faz sentido, ou strings como `'Not available'`).

In [ ]:
# 5.1 Verificação de nulos explícitos
print('Valores nulos por coluna:')
print(df.isnull().sum())
print(f'\nTotal de nulos no dataset: {df.isnull().sum().sum()}')

In [ ]:
# 5.2 Verificação de dados faltantes IMPLÍCITOS
# Valores não documentados que podem representar dados ausentes

print('⚠️  Investigação de dados faltantes implícitos:\n')

# PAY_1 com 'Not available'
pay1_na = (df['PAY_1'] == 'Not available').sum()
print(f"PAY_1 com 'Not available': {pay1_na} ocorrências")

# EDUCATION com valor 0 (não documentado)
edu_zero = (df['EDUCATION'] == 0).sum()
print(f"EDUCATION com valor 0 (não documentado): {edu_zero} ocorrências")

# MARRIAGE com valor 0 (não documentado)
mar_zero = (df['MARRIAGE'] == 0).sum()
print(f"MARRIAGE com valor 0 (não documentado): {mar_zero} ocorrências")

print('\n📌 Conclusão: Não há nulos explícitos, mas existem valores não documentados')
print('   que precisarão de tratamento nas próximas etapas do projeto.')

---

## ✅ Resumo da Exploração de Dados

| Pergunta | Resposta |
|---|---|
| Quantas colunas? | **25** (1 metadado, 23 características, 1 resposta) |
| Quantas linhas? | **30.000** (uma por conta de crédito) |
| Características categóricas | `SEX`, `EDUCATION`, `MARRIAGE`, `PAY_1` a `PAY_6` |
| Características numéricas | `LIMIT_BAL`, `AGE`, `BILL_AMT1–6`, `PAY_AMT1–6` |
| Nulos explícitos? | **Não** — zero valores nulos |
| Nulos implícitos? | **Sim** — `PAY_1` com `'Not available'`, `EDUCATION` e `MARRIAGE` com valor `0` |

**Próximos passos:** Limpeza e tratamento dos dados — remover ou corrigir os valores problemáticos encontrados.